# Stage 3.C — fine-tier 3D verification of the campaign's top designs

Fine-verifies the top designs from the finished **Stage-3 trapezoid campaign** at the high-fidelity 3D tier (5 cycles / 60 inner-iter vs the campaign's coarse 3/30). It reads the campaign's Drive shards (`campaign_trapezoid/evaluations_*.jsonl`), ranks by coarse J_fan, and re-runs the top-K — catching designs whose coarse score is a low-fidelity **artifact** (they diverge when run longer) and giving the trustworthy final aero numbers to pick V1 from. Then `recommend_designs` prints the V1 pick.

**Resumable:** `verification.json` is written to Drive after *each* design; a Colab drop → just re-run the RUN cell and it skips the ones already done and finishes the rest. Fine runs are ~10-12 h/design, parallel across `--workers`, so budget ~a day for 20 on one session.

**Run order:** cell 1 (connect Drive) → cell 2 (repo+deps) → cell 3 (SU2) → config → preview → RUN → pick.

## 1. Connect Drive

In [ ]:
# Drive connect ONLY (kept separate from the repo/deps install below).
import importlib.util
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/fanopt")
else:
    DRIVE_ROOT = Path.cwd() / "data"
print("drive root:", DRIVE_ROOT)

## 2. Repo + deps  (separate cell from the Drive connect above)

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")   # 1 thread/worker -> N processes on N cores

IN_COLAB = importlib.util.find_spec("google.colab") is not None
BRANCH = "main"  # the 3.C verify tool + this notebook are on main
REPO = Path("/content/fan-optimization") if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH,
                        "https://github.com/clingergab/fan-optimization.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "origin", BRANCH], check=True)
    subprocess.run("apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 "
                   "libxft2 libxinerama1 unzip".split(), check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[bo]"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gmsh", "cadquery"], check=True)
for p in (str(REPO), str(REPO / "src"), str(REPO / "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo:", REPO)

## 3. SU2 solver

In [ ]:
import subprocess, urllib.request
from pathlib import Path
from fanopt.cfd.phase3 import find_su2
SU2_BIN = find_su2()
if SU2_BIN is None and IN_COLAB:
    LOCAL = Path("/content/su2")
    if not any(LOCAL.rglob("SU2_CFD")):
        zc = DRIVE_ROOT / "su2" / "SU2-v8.0.1-linux64.zip"
        if not zc.exists():
            zc.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(
                "https://github.com/su2code/SU2/releases/download/v8.0.1/SU2-v8.0.1-linux64.zip", str(zc))
        LOCAL.mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", str(zc), "-d", str(LOCAL)], check=True)
    hit = next(LOCAL.rglob("SU2_CFD"), None)
    if hit: subprocess.run(["chmod", "+x", str(hit)], check=False)
    SU2_BIN = str(hit) if hit else None
assert SU2_BIN, "SU2 not found"
print("SU2:", SU2_BIN)

## 4. Config — point at the finished campaign

In [ ]:
import os
from fanopt.cfd.phase5 import FINE_CYCLES, FINE_INNER
# ---- the SAME folder the Stage-3 campaign wrote to ----
SHARED_DIR = DRIVE_ROOT / "campaign_trapezoid"
OUT_DIR    = DRIVE_ROOT / "phase5_verify_blade"   # verification.json lands here (Drive => resumable)
TOP_K      = 20          # fine-verify the top-20 by coarse J_fan (hedge vs fine-tier failures)
TO_PROMOTE = 10          # of the fine-verified, how many (top by fine J_fan) to promote to TO
N_WORKERS  = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
N_CYCLES   = FINE_CYCLES  # the FINE tier (5 cycles / 60 inner) — the 3.C fidelity, ~10-12 h/design
INNER_ITER = FINE_INNER
assert SHARED_DIR.exists(), f"campaign folder not found: {SHARED_DIR}"
n_shards = len(list(SHARED_DIR.glob("evaluations_*.jsonl")))
assert n_shards, f"no evaluations_*.jsonl shards in {SHARED_DIR} - is the folder right?"
print(f"campaign: {SHARED_DIR}  ({n_shards} shards)")
print(f"fine tier: {N_CYCLES} cycles / {INNER_ITER} inner | top-{TOP_K} | {N_WORKERS} workers | out {OUT_DIR}")

## 5. Preview — the top-K designs that will be verified (sanity: right folder?)

In [ ]:
from fanopt.bo.distributed_campaign import read_ledger
from fanopt.cfd.blade_verify import top_designs_from_shards
n_unique = len(read_ledger(SHARED_DIR)[0])
designs = top_designs_from_shards(SHARED_DIR, top_k=TOP_K)
print(f"campaign has {n_unique} unique designs; will fine-verify the top {len(designs)} by coarse J_fan:\n")
print(f"  {'#':>2} {'coarse J_fan':>13}  {'rib':8} design_hash")
for i, (name, params, j_coarse) in enumerate(designs):
    rib = "uniform" if params.uniform else "ribbed"
    print(f"  {i:>2} {j_coarse:+.3e}  {rib:8} {name.split('_', 1)[1][:20]}")

## 6. RUN the fine-verify  (resumable — re-run after a drop to continue)

In [ ]:
import fanopt.geometry.blade_cad as blade_cad
import run_phase5_verify_blade as verify
from fanopt.cfd.phase5 import VerifyConfig
blade_cad.N_RADIAL_SECTIONS = 40   # match the campaign objective's geometry resolution (ADR-0004)
# verification.json is written to Drive after EACH design. A Colab drop -> just re-run THIS cell;
# it keeps the finite results already there and re-verifies only the rest.
cfg = VerifyConfig(n_cycles=N_CYCLES, inner_iter=INNER_ITER)
summary = verify.run(out_dir=OUT_DIR, top_k=TOP_K, shared_dir=SHARED_DIR, su2_bin=SU2_BIN,
                     cfg=cfg, n_workers=N_WORKERS, progress=True)
r = summary["ranking"]
print(f"\nverified {len(summary['designs'])} designs | rank_preserved={r['rank_preserved']} "
      f"(tau={r['kendall_tau']}, n={r['n']}, suspect={r['n_suspect']})")

## 7. Rank the verified designs — mark the top ~10 to promote to TO


In [ ]:
import recommend_designs as recommend
# Joins the fine verification.json to the campaign shards by design_hash and ranks by fine J_fan.
# The full `ranked` table = every verified design; * marks the top-`TO_PROMOTE` structurally-diverse
# designs to PROMOTE TO TO (Phase 2). The final 3-to-print set is chosen AFTER TO, not here.
recommend.main(["--shared-dir", str(SHARED_DIR),
                "--verification", str(OUT_DIR / "verification.json"),
                "--out-dir", str(OUT_DIR), "--top-k", str(TO_PROMOTE)])

## 8. Render the top 5 (by fine J_fan) in 3D

In [ ]:
import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import fanopt.geometry.blade_cad as blade_cad
from fanopt.geometry.blade_cad import blade_trimesh
from fanopt.bo.blade_codec import decode
from fanopt.bo.campaign_analysis import classify_panel
blade_cad.N_RADIAL_SECTIONS = 40   # match the campaign/verify geometry resolution (ADR-0003)

N_TOP  = 5    # top designs by FINE whole-fan J_fan to render
N_COLS = 2    # renders per row; the grid grows DOWN so you scroll through them

# top-N by FINE J_fan, from the verification.json this notebook wrote
ver = json.loads((OUT_DIR / "verification.json").read_text())
fine = [d for d in ver["designs"]
        if isinstance(d.get("j_fan_3d"), (int, float)) and np.isfinite(d["j_fan_3d"])]
fine = sorted(fine, key=lambda d: -d["j_fan_3d"])[:N_TOP]
assert fine, "no verified designs yet — run the RUN cell (cell 6) first."

# design_hash -> shard row (for the vector to rebuild the blade); verify names are '{rank}_{hash}'
by_hash = {}
for _f in SHARED_DIR.glob("evaluations_*.jsonl"):
    for _l in _f.read_text().splitlines():
        if _l.strip():
            _d = json.loads(_l)
            by_hash[_d.get("design_hash")] = _d

meshes, titles = [], []
for k, d in enumerate(fine):
    row = by_hash[str(d["name"]).split("_", 1)[1]]   # hash suffix -> the campaign shard row
    params = decode(np.array(row["vector"], dtype=float))
    v, faces = blade_trimesh(params, tol=0.001)
    meshes.append((v, faces))
    rib = "uniform" if params.uniform else "ribbed"
    titles.append(f"#{k+1} fine J={d['j_fan_3d']:+.2e}<br>{row['mass_kg']*1e3:.0f}g \u00b7 {rib} \u00b7 "
                  f"panel:{classify_panel(params.panel_offsets_m)}")

# Plotly Mesh3d => DRAG to rotate, SCROLL to zoom. aspectmode='data' gives TRUE proportions
# (the rib bow is a gentle rise on a 220 mm blade, not an exaggerated wedge).
n_rows = -(-len(meshes) // N_COLS)   # ceil: enough rows for every design
fig = make_subplots(rows=n_rows, cols=N_COLS,
                    specs=[[{"type": "scene"}] * N_COLS for _ in range(n_rows)],
                    subplot_titles=titles, horizontal_spacing=0.02, vertical_spacing=0.05)
for k, (v, faces) in enumerate(meshes):
    fig.add_trace(go.Mesh3d(x=v[:, 0], y=v[:, 1], z=v[:, 2],
                            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                            intensity=v[:, 2], colorscale="Viridis", showscale=False),
                  row=k // N_COLS + 1, col=k % N_COLS + 1)
fig.for_each_scene(lambda s: s.update(aspectmode="data", xaxis_visible=False,
                                      yaxis_visible=False, zaxis_visible=False))
fig.update_layout(height=430 * n_rows, margin=dict(l=0, r=0, t=70, b=0),
                  title_text=f"Top {len(meshes)} by FINE whole-fan J_fan \u2014 DRAG to rotate, SCROLL to zoom")
fig.show()